In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

##Load Silver

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_TABLE = "workspace.silver.transactions"

silver_df = spark.table(SILVER_TABLE)

print(
    "Silver transactions:",
    f"{silver_df.count():,}"
)

Silver transactions: 4,681,541


##Account Balance

In [0]:
balance_df = (
    silver_df
    .withColumn(
        "signed_amount",
        F.when(
            F.col("txn_type") == "CREDIT",
            F.col("amount")
        )
        .when(
            F.col("txn_type") == "DEBIT",
            -F.col("amount")
        )
        .otherwise(F.lit(0))
    )
)

##Calculate current balance

In [0]:
account_balances = (
    balance_df
    .groupBy("account_id")
    .agg(
        F.round(
            F.sum("signed_amount"),
            2
        ).cast("decimal(18,2)").alias("current_balance"),

        F.max("txn_time").alias("last_updated"),

        F.count("*").alias("transaction_count")
    )
)

##Check balances

In [0]:
print(
    "Accounts with transactions:",
    f"{account_balances.count():,}"
)

display(
    account_balances
    .orderBy(F.desc("current_balance"))
    .limit(20)
)

Accounts with transactions: 135,058


account_id,current_balance,last_updated,transaction_count
ACC029959,3681750.35,2025-12-26T00:43:19.000Z,36
ACC016396,3589106.49,2025-12-08T01:53:40.000Z,35
ACC113719,3446929.10,2025-12-10T08:03:25.000Z,36
ACC121401,3380166.87,2025-12-31T08:40:24.000Z,32
ACC007110,3356982.86,2025-12-31T05:59:03.000Z,34
ACC065337,3329837.44,2025-12-28T04:59:28.000Z,35
ACC064140,3323933.23,2025-12-22T03:43:18.000Z,36
ACC052262,3293871.09,2025-12-30T08:31:37.000Z,37
ACC051080,3291082.16,2025-12-27T01:08:43.000Z,36
ACC078337,3281343.85,2025-12-15T06:18:03.000Z,35


##Save Gold Account Balances

In [0]:
(
    account_balances
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.account_balances")
)

print("Gold account balances created")

Gold account balances created


##Fraud Detection: High Value

In [0]:
high_value_alerts = (
    silver_df
    .filter(F.col("amount") >= 100_000)
    .withColumn(
        "risk_type",
        F.lit("HIGH_VALUE_TRANSACTION")
    )
    .withColumn(
        "risk_score",
        F.when(F.col("amount") >= 500_000, 95)
         .when(F.col("amount") >= 250_000, 85)
         .otherwise(70)
    )
)

In [0]:
print(
    "High-value transactions:",
    f"{high_value_alerts.count():,}"
)

display(
    high_value_alerts
    .select(
        "txn_id",
        "account_id",
        "amount",
        "channel",
        "txn_time",
        "risk_type",
        "risk_score"
    )
    .orderBy(F.desc("amount"))
    .limit(20)
)

High-value transactions: 1,619,023


txn_id,account_id,amount,channel,txn_time,risk_type,risk_score
TXN0003331690,ACC100336,999994.42,NETBANKING,2025-11-04T16:00:49.000Z,HIGH_VALUE_TRANSACTION,95
TXN0000511882,ACC118565,999984.16,NETBANKING,2025-01-27T16:36:41.000Z,HIGH_VALUE_TRANSACTION,95
TXN0001556786,ACC079140,999955.8,UPI,2025-10-23T17:03:47.000Z,HIGH_VALUE_TRANSACTION,95
TXN0001883140,ACC141464,999906.45,UPI,2025-10-13T17:58:59.000Z,HIGH_VALUE_TRANSACTION,95
TXN0002533295,ACC113614,999897.39,NETBANKING,2025-10-25T16:28:02.000Z,HIGH_VALUE_TRANSACTION,95
TXN0003568178,ACC063018,999889.23,NETBANKING,2025-12-21T02:21:13.000Z,HIGH_VALUE_TRANSACTION,95
TXN0002250096,ACC099085,999869.66,UPI,2025-03-10T13:32:18.000Z,HIGH_VALUE_TRANSACTION,95
TXN0002562159,ACC145585,999817.85,BRANCH,2025-04-13T07:33:42.000Z,HIGH_VALUE_TRANSACTION,95
TXN0002491596,ACC067345,999805.87,NETBANKING,2025-09-11T16:26:06.000Z,HIGH_VALUE_TRANSACTION,95
TXN0000083658,ACC092968,999792.77,UPI,2025-01-28T18:31:56.000Z,HIGH_VALUE_TRANSACTION,95


##Velocity Fraud Detection

In [0]:
velocity_window = (
    Window
    .partitionBy("account_id")
    .orderBy(F.col("txn_time").cast("long"))
    .rangeBetween(-600, 0)
)

velocity_df = (
    silver_df
    .withColumn(
        "txn_count_10min",
        F.count("*").over(velocity_window)
    )
)

print("10-minute velocity calculation completed")

10-minute velocity calculation completed


In [0]:
velocity_alerts = (
    velocity_df
    .filter(F.col("txn_count_10min") >= 5)
    .withColumn(
        "risk_type",
        F.lit("HIGH_VELOCITY")
    )
    .withColumn(
        "risk_score",
        F.when(
            F.col("txn_count_10min") >= 10,
            95
        )
        .when(
            F.col("txn_count_10min") >= 7,
            85
        )
        .otherwise(75)
    )
)

##Channel behavior

In [0]:
channel_summary = (
    silver_df
    .groupBy(
        "account_id",
        "channel"
    )
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(
            F.sum("amount"),
            2
        ).cast("decimal(18,2)").alias("total_amount")
    )
)

##Create fraud alert table

In [0]:
high_value_fraud = (
    high_value_alerts
    .select(
        "txn_id",
        "account_id",
        "customer_id",
        "amount",
        "channel",
        "txn_time",
        "risk_type",
        "risk_score"
    )
)

velocity_fraud = (
    velocity_alerts
    .select(
        "txn_id",
        "account_id",
        "customer_id",
        "amount",
        "channel",
        "txn_time",
        "risk_type",
        "risk_score"
    )
)

fraud_alerts = (
    high_value_fraud
    .unionByName(velocity_fraud)
    .dropDuplicates(["txn_id", "risk_type"])
)

##Save fraud alerts

In [0]:
(
    fraud_alerts
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.fraud_alerts")
)

print("Gold fraud alerts created")

Gold fraud alerts created


##Daily Banking KPIs

In [0]:
daily_metrics = (
    silver_df
    .withColumn(
        "txn_date",
        F.to_date("txn_time")
    )
    .groupBy("txn_date")
    .agg(
        F.count("*").alias("transaction_count"),

        F.round(
            F.sum("amount"),
            2
        ).cast("decimal(18,2)").alias("total_transaction_amount"),

        F.round(
            F.avg("amount"),
            2
        ).cast("decimal(18,2)").alias("average_transaction_amount"),

        F.countDistinct("account_id")
         .alias("active_accounts"),

        F.countDistinct("customer_id")
         .alias("active_customers"),

        F.sum(
            F.when(
                F.col("txn_type") == "CREDIT",
                F.col("amount")
            ).otherwise(0)
        ).alias("credit_amount"),

        F.sum(
            F.when(
                F.col("txn_type") == "DEBIT",
                F.col("amount")
            ).otherwise(0)
        ).alias("debit_amount")
    )
)

##Save daily metrics

In [0]:
(
    daily_metrics
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.daily_transaction_metrics")
)

print("Gold daily metrics created")

Gold daily metrics created
